# Methods and results

`nearest` takes the nearest node and time step, `interp` weights several of them.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from floatmatcher.floatmatcher import FloatMatcher

### Points and product

In [ ]:
df = pd.DataFrame({
    "longitude": [-38.2, -35.6, -31.9, -27.4, -23.1],
    "latitude":  [31.5, 34.2, 37.8, 40.1, 43.6],
    "date": pd.to_datetime(["2018-01-02 03:10", "2018-01-03 14:45", "2018-01-05 08:20",
                            "2018-01-06 21:05", "2018-01-08 11:30"]),
})

my_fm = FloatMatcher()
my_fm.set_points(df)
my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"],
                  store="/runtime/data/era5-hourly",  # adapt to your archive
                  pattern="{year}/{month:02d}/era5_single-levels_{year}{month:02d}{day:02d}.nc",
                  )

### Nearest

In [ ]:
my_fm.set_method(type="nearest", radius=25, time_gap=np.timedelta64(12, "h"))
my_fm.match()
my_fm.result

In [ ]:
# one value per point, with its distance to the node and its gap to the time step
result = my_fm.result
pd.DataFrame({
    "sst": result.values["sst"],
    "distance_km": result.distance_km,
    "time_gap_h": result.time_delta / 3600,
    "valid": result.valid,
})

In [ ]:
nearest = my_fm.to_dataframe()  # the input DataFrame, with sst_coloc added

### Interpolation

In [ ]:
my_fm.set_method(type="interp",
                 radius=50,
                 time_gap=np.timedelta64(12, "h"),
                 k_nearest=4,     # nodes retained for interpolation
                 n_frames=2,      # time steps weightused for interpolation
                 kernel="idw",    # inverse distance weighting
                 power=2,
                 )
my_fm.match()
interp = my_fm.to_dataframe()

pd.DataFrame({"nearest": nearest["sst_coloc"], "interp": interp["sst_coloc"]})

### Radius and time gap

Beyond `radius` or `time_gap`, a point is not matched: NaN value and `valid=False`. Its distance and time gap are kept in `result`, to see how far it was.

In [ ]:
my_fm.set_method(type="nearest", radius=12, time_gap=np.timedelta64(4, "h"))
my_fm.match()
my_fm.to_dataframe()  # points 3 and 4 are more than 12 km from a node

### Several neighbors

`k_nearest` nodes and `n_frames` time steps per point, sorted from the nearest: `s0` is the nearest node, `t0` the nearest time step.

In [ ]:
my_fm.set_method(type="nearest", radius=50, time_gap=np.timedelta64(12, "h"),
                 k_nearest=2, n_frames=2)
my_fm.match()
print(my_fm.result.values["sst"].shape)   # (5, 2, 2): points x nodes x time steps
my_fm.to_dataframe()                          # one column per neighbor: sst_coloc_s0_t0, sst_coloc_s0_t1...

For a Dataset, the neighbors are the `k_space` and `n_time` dimensions, or one variable per neighbor with `neigh_to_vars=True`.

In [ ]:
ds = xr.Dataset({"LONGITUDE": ("N_PROF", df["longitude"].values),
                 "LATITUDE": ("N_PROF", df["latitude"].values),
                 "TIME": ("N_PROF", df["date"].values)})
my_fm.set_points(ds)
my_fm.match()
my_fm.to_xarray()                          # sst_coloc(N_PROF, k_space, n_time)

In [ ]:
my_fm.to_xarray(neigh_to_vars=True)        # sst_coloc_s0_t0(N_PROF), sst_coloc_s0_t1(N_PROF)...

In [ ]:
# attributes of the gridded variable are always kept
print(my_fm.to_xarray().sst_coloc.attrs)

# in_place=True adds the matched variables to the input Dataset instead of a copy
my_fm.to_xarray(in_place=True)
print("sst_coloc" in ds)   # True